In [8]:
import mlflow

mlflow.set_experiment("Hyperparameter Tuning Experiment")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1793972439637, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1793972439637, lifecycle_stage='active', name='Hyperparameter Tuning Experiment', tags={}, trace_location=None, workspace='default'>

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

In [10]:
import mlflow
import optuna
import sklearn


def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        rf_max_depth = trial.suggest_int("rf_max_depth", 2, 32)
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300, step=10)
        rf_max_features = trial.suggest_float("rf_max_features", 0.2, 1.0)
        params = {
            "max_depth": rf_max_depth,
            "n_estimators": rf_n_estimators,
            "max_features": rf_max_features,
        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = sklearn.ensemble.RandomForestRegressor(**params)
        regressor_obj.fit(X_train, y_train)

        y_pred = regressor_obj.predict(X_val)
        error = sklearn.metrics.mean_squared_error(y_val, y_pred)
        # Log current trial's error metric
        mlflow.log_metrics({"error": error})

        # Log the model file. MLflow saves sklearn models with skops by default,
        # which refuses tree-based models unless the Tree type is explicitly trusted.
        # Safe here because we trained this model ourselves.
        mlflow.sklearn.log_model(
            regressor_obj,
            name="model",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        return error
    
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="study") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-11-06 22:16:08,500] A new study created in memory with name: no-name-56135aa3-62f3-4b93-b0fa-bed21bd13c91


🏃 View run trial_0 at: http://localhost:5000/#/experiments/3/runs/098637f3102142eb99bbb87efc6ad57a
🧪 View experiment at: http://localhost:5000/#/experiments/3


[I 2026-11-06 22:16:31,509] Trial 0 finished with value: 0.30368158328313655 and parameters: {'rf_max_depth': 10, 'rf_n_estimators': 220, 'rf_max_features': 0.8116522333881195}. Best is trial 0 with value: 0.30368158328313655.


🏃 View run trial_1 at: http://localhost:5000/#/experiments/3/runs/c404411bd7aa464f9478a834e2b4408f
🧪 View experiment at: http://localhost:5000/#/experiments/3


[I 2026-11-06 22:16:45,117] Trial 1 finished with value: 0.26694806940982696 and parameters: {'rf_max_depth': 14, 'rf_n_estimators': 230, 'rf_max_features': 0.7429848494566396}. Best is trial 1 with value: 0.26694806940982696.
[I 2026-11-06 22:16:58,674] Trial 2 finished with value: 0.24468034746842482 and parameters: {'rf_max_depth': 32, 'rf_n_estimators': 240, 'rf_max_features': 0.4340298126476594}. Best is trial 2 with value: 0.24468034746842482.


🏃 View run trial_2 at: http://localhost:5000/#/experiments/3/runs/84cf8fbcc916434dae632d02d50d6228
🧪 View experiment at: http://localhost:5000/#/experiments/3


... (40 outputs left out)

[I 2026-11-06 22:22:55,397] Trial 27 finished with value: 0.2500308761477678 and parameters: {'rf_max_depth': 27, 'rf_n_estimators': 100, 'rf_max_features': 0.38350880586349845}. Best is trial 22 with value: 0.24271644740333764.


🏃 View run trial_27 at: http://localhost:5000/#/experiments/3/runs/b882dc45ac3041f68d90853ed103582d
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run trial_28 at: http://localhost:5000/#/experiments/3/runs/c65e4007010c402fa2ebd740fcfc04b5
🧪 View experiment at: http://localhost:5000/#/experiments/3


[I 2026-11-06 22:23:08,888] Trial 28 finished with value: 0.2834821469504601 and parameters: {'rf_max_depth': 22, 'rf_n_estimators': 220, 'rf_max_features': 0.2091957527596326}. Best is trial 22 with value: 0.24271644740333764.
[I 2026-11-06 22:23:17,038] Trial 29 finished with value: 0.2550350833700654 and parameters: {'rf_max_depth': 17, 'rf_n_estimators': 100, 'rf_max_features': 0.3404140941702949}. Best is trial 22 with value: 0.24271644740333764.


🏃 View run trial_29 at: http://localhost:5000/#/experiments/3/runs/ed77312e197043c9bb342a2498328652
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run study at: http://localhost:5000/#/experiments/3/runs/bc22b3717ae44768b4a953b71128b451
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [1]:
import mlflow

best_run_id = "d5c1e48e4dd9400faac6af66991783ab"
model_uri = f"runs:/{best_run_id}/model"

mlflow.register_model(
    model_uri = model_uri,
    name = "housing-price-predictor"
)


/home/hajar/explore/mlflow-recap/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Successfully registered model 'housing-price-predictor'.
2026/11/07 10:57:45 WARNING mlflow.tracking._model_registry.fluent: Run with id d5c1e48e4dd9400faac6af66991783ab has no artifacts at artifact path 'model', registering model based on models:/m-cfffe38c60ad484da2662d88bef04cf6 instead
2026/11/07 10:57:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: housing-price-predictor, version 1
Created version '1' of model 'housing-price-predictor'.


<ModelVersion: aliases=[], creation_timestamp=1794020265305, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1794020265305, metrics=None, model_id=None, name='housing-price-predictor', params=None, run_id='d5c1e48e4dd9400faac6af66991783ab', run_link='', source='models:/m-cfffe38c60ad484da2662d88bef04cf6', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>